# Week-07: Naive Bayes Classifier — Solutions

From-scratch implementations only (no scikit-learn).

**Contents**
1. Bayes theorem (Q1a, Q1b)
2. Categorical Naive Bayes class
3. Buy-computer dataset (Q2)
4. Sports text classifier (Q3)
5. Additional Q1 — Outlook → Play
6. Additional Q2 — Play golf (4 features)
7. Additional Q3 — Sentiment text

## Shared utilities

In [ ]:
from collections import Counter
import math
import csv
from io import StringIO


def bayes(prior_h, lik_given_h, evidence):
    return (prior_h * lik_given_h) / evidence


def load_csv_str(s):
    return list(csv.DictReader(StringIO(s.strip())))


def metrics(y_true, y_pred):
    n = len(y_true)
    acc = sum(t == p for t, p in zip(y_true, y_pred)) / n if n else 0.0
    labels = sorted(set(y_true) | set(y_pred))
    per = {}
    for lab in labels:
        tp = sum(t == lab and p == lab for t, p in zip(y_true, y_pred))
        fp = sum(t != lab and p == lab for t, p in zip(y_true, y_pred))
        fn = sum(t == lab and p != lab for t, p in zip(y_true, y_pred))
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        per[lab] = {"precision": prec, "recall": rec, "support": sum(t == lab for t in y_true)}
    return {"accuracy": acc, "per_class": per}


def fmt_post(post):
    return {k: round(v, 4) for k, v in post.items()}


## Q1a — Probability the student is a hosteler given A grade

- P(H) = 0.6, P(D) = 0.4
- P(A|H) = 0.3, P(A|D) = 0.2

P(H|A) = P(A|H) P(H) / P(A)

In [ ]:
p_hostel, p_day = 0.60, 0.40
p_a_hostel, p_a_day = 0.30, 0.20
p_a = p_a_hostel * p_hostel + p_a_day * p_day
p_hostel_given_a = bayes(p_hostel, p_a_hostel, p_a)

print(f"P(A)          = {p_a:.4f}")
print(f"P(hostel | A) = {p_hostel_given_a:.6f}  ({p_hostel_given_a*100:.2f}%)")


**Answer:** P(hosteler | A) ≈ **0.6923 (69.23%)**.

## Q1b — Disease given a positive test

- Prevalence P(D) = 0.01
- Sensitivity P(+|D) = 0.99
- False-positive rate P(+|not D) = 0.02

In [ ]:
p_d, p_not_d = 0.01, 0.99
p_pos_d, p_pos_not_d = 0.99, 0.02
p_pos = p_pos_d * p_d + p_pos_not_d * p_not_d
p_d_given_pos = bayes(p_d, p_pos_d, p_pos)

print(f"P(+)      = {p_pos:.4f}")
print(f"P(D | +)  = {p_d_given_pos:.6f}  ({p_d_given_pos*100:.2f}%)")


**Answer:** P(D | +) = **1/3 ≈ 0.3333 (33.33%)**.

## Categorical Naive Bayes (from scratch)

Laplace-smoothed class-conditional probabilities:

P(x_j = v | C) = (n_{C,v} + α) / (n_C + α |V_j|)

In [ ]:
class CategoricalNB:
    """Multinomial / categorical Naive Bayes with Laplace smoothing."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y, feature_names=None):
        self.n_ = len(y)
        self.classes_ = sorted(set(y))
        self.class_counts_ = Counter(y)
        if feature_names is None:
            if isinstance(X[0], dict):
                self.feature_names_ = list(X[0].keys())
                rows = X
            else:
                raise ValueError("feature_names required when X is list-of-lists")
        else:
            self.feature_names_ = list(feature_names)
            rows = [dict(zip(self.feature_names_, row)) for row in X]

        self.feature_counts_ = {f: {c: Counter() for c in self.classes_} for f in self.feature_names_}
        self.feature_values_ = {f: set() for f in self.feature_names_}
        for row, label in zip(rows, y):
            for f in self.feature_names_:
                v = row[f]
                self.feature_counts_[f][label][v] += 1
                self.feature_values_[f].add(v)
        return self

    def _log_prob(self, row):
        scores = {}
        for c in self.classes_:
            lp = math.log(self.class_counts_[c] / self.n_)
            for f in self.feature_names_:
                v = row[f]
                n_cv = self.feature_counts_[f][c][v]
                n_c = self.class_counts_[c]
                V = len(self.feature_values_[f])
                p = (n_cv + self.alpha) / (n_c + self.alpha * V)
                lp += math.log(p)
            scores[c] = lp
        return scores

    def predict_one(self, row):
        if not isinstance(row, dict):
            row = dict(zip(self.feature_names_, row))
        scores = self._log_prob(row)
        m = max(scores.values())
        exps = {c: math.exp(s - m) for c, s in scores.items()}
        z = sum(exps.values())
        post = {c: exps[c] / z for c in self.classes_}
        pred = max(post, key=post.get)
        return pred, post

    def predict(self, X):
        return [self.predict_one(r)[0] for r in X]


## Text Naive Bayes (bag of words)

In [ ]:
class TextNB:
    """Multinomial Naive Bayes over word counts, Laplace smoothing."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def _tokenize(self, text):
        return [w.lower() for w in text.replace("'", " ").split() if w.strip()]

    def fit(self, texts, labels):
        self.classes_ = sorted(set(labels))
        self.class_counts_ = Counter(labels)
        self.n_ = len(labels)
        self.word_counts_ = {c: Counter() for c in self.classes_}
        self.total_words_ = {c: 0 for c in self.classes_}
        vocab = set()
        for t, lab in zip(texts, labels):
            toks = self._tokenize(t)
            self.word_counts_[lab].update(toks)
            self.total_words_[lab] += len(toks)
            vocab.update(toks)
        self.vocab_ = sorted(vocab)
        self.V_ = len(self.vocab_)
        return self

    def _log_prob(self, text):
        toks = self._tokenize(text)
        scores = {}
        for c in self.classes_:
            lp = math.log(self.class_counts_[c] / self.n_)
            den = self.total_words_[c] + self.alpha * self.V_
            for w in toks:
                num = self.word_counts_[c][w] + self.alpha
                lp += math.log(num / den)
            scores[c] = lp
        return scores

    def predict_one(self, text):
        scores = self._log_prob(text)
        m = max(scores.values())
        exps = {c: math.exp(s - m) for c, s in scores.items()}
        z = sum(exps.values())
        post = {c: exps[c] / z for c in self.classes_}
        pred = max(post, key=post.get)
        return pred, post

    def predict(self, texts):
        return [self.predict_one(t)[0] for t in texts]


## Q2 — Should the buyer buy a computer?

Classic categorical dataset (age, income, student, credit_rating → buys_computer).

In [ ]:
BUY_COMPUTER_CSV = """age,income,student,credit_rating,buys_computer
<=30,high,no,fair,no
<=30,high,no,excellent,no
31...40,high,no,fair,yes
>40,medium,no,fair,yes
>40,low,yes,fair,yes
>40,low,yes,excellent,no
31...40,low,yes,excellent,yes
<=30,medium,no,fair,no
<=30,low,yes,fair,yes
>40,medium,yes,fair,yes
<=30,medium,yes,excellent,yes
31...40,medium,no,excellent,yes
31...40,high,yes,fair,yes
>40,medium,no,excellent,no
"""

rows = load_csv_str(BUY_COMPUTER_CSV)
feats = ["age", "income", "student", "credit_rating"]
X = [{f: r[f] for f in feats} for r in rows]
y = [r["buys_computer"] for r in rows]

clf = CategoricalNB(alpha=1.0).fit(X, y)
query = {"age": "<=30", "income": "medium", "student": "yes", "credit_rating": "fair"}
pred, post = clf.predict_one(query)
yhat = clf.predict(X)
m = metrics(y, yhat)

print("N =", len(y), " class counts =", dict(clf.class_counts_))
print("Query:", query)
print("Prediction: buys_computer =", pred)
print("Posterior:", fmt_post(post))
print("Train accuracy:", round(m["accuracy"], 3))
print("Per class:", m["per_class"])


**Answer:** for `<=30 / medium / student=yes / fair` → **yes** (posterior ≈ 0.77). Train accuracy 92.9%.

## Q3 — Sports vs not-sports text

Classify the sentence **"A very close game"**.

In [ ]:
SPORTS_DATA = [
    ("A great game", "Sports"),
    ("The election was over", "Not sports"),
    ("Very clean match", "Sports"),
    ("A clean but forgettable game", "Sports"),
    ("It was a close election", "Not sports"),
]

texts = [t for t, _ in SPORTS_DATA]
labels = [l for _, l in SPORTS_DATA]
clf = TextNB(alpha=1.0).fit(texts, labels)

query = "A very close game"
pred, post = clf.predict_one(query)
yhat = clf.predict(texts)

print("Vocab size:", clf.V_, " class counts:", dict(clf.class_counts_))
print("Query:", query)
print("Tag:", pred)
print("Posterior:", fmt_post(post))
print("Train accuracy:", metrics(labels, yhat)["accuracy"])


**Answer:** **Sports** (posterior ≈ 0.83).

## Additional Q1 — Outlook only

Classify: if weather is **Sunny**, should the player play?

In [ ]:
OUTLOOK_CSV = """outlook,play
Rainy,Yes
Sunny,Yes
Overcast,Yes
Overcast,Yes
Sunny,No
Rainy,Yes
Sunny,Yes
Overcast,Yes
Rainy,No
Sunny,No
Sunny,Yes
Rainy,No
Overcast,Yes
Overcast,Yes
"""

rows = load_csv_str(OUTLOOK_CSV)
X = [{"outlook": r["outlook"]} for r in rows]
y = [r["play"] for r in rows]
clf = CategoricalNB(alpha=1.0).fit(X, y)

query = {"outlook": "Sunny"}
pred, post = clf.predict_one(query)
yhat = clf.predict(X)
m = metrics(y, yhat)

print("N =", len(y), " class counts =", dict(clf.class_counts_))
print("Query:", query)
print("Prediction: play =", pred)
print("Posterior:", fmt_post(post))
print("Train accuracy:", round(m["accuracy"], 3))
print("Per class:", m["per_class"])


**Answer:** **Play = Yes** (P(Yes|Sunny) ≈ 0.64). Single-feature model never predicts No, so train accuracy is 71.4%.

## Additional Q2 — Play golf (weather, temperature, humidity, windy)

In [ ]:
GOLF_CSV = """weather,temperature,humidity,windy,play
Rainy,Hot,High,False,No
Rainy,Hot,High,True,No
Overcast,Hot,High,False,Yes
Sunny,Mild,High,False,Yes
Sunny,Cool,Normal,False,Yes
Sunny,Cool,Normal,True,No
Overcast,Cool,Normal,True,Yes
Rainy,Mild,High,False,No
Sunny,Cool,Normal,False,Yes
Rainy,Mild,Normal,False,Yes
Sunny,Mild,Normal,False,Yes
Overcast,Mild,High,True,Yes
Overcast,Hot,Normal,False,Yes
Sunny,Mild,High,True,No
"""

rows = load_csv_str(GOLF_CSV)
feats = ["weather", "temperature", "humidity", "windy"]
X = [{f: r[f] for f in feats} for r in rows]
y = [r["play"] for r in rows]
clf = CategoricalNB(alpha=1.0).fit(X, y)

queries = [
    {"weather": "Sunny", "temperature": "Hot", "humidity": "High", "windy": "False"},
    {"weather": "Sunny", "temperature": "Cool", "humidity": "High", "windy": "True"},
    {"weather": "Overcast", "temperature": "Hot", "humidity": "High", "windy": "False"},
]

print("N =", len(y), " class counts =", dict(clf.class_counts_))
for q in queries:
    pred, post = clf.predict_one(q)
    print("Query", q, "→ play =", pred, " post =", fmt_post(post))

yhat = clf.predict(X)
m = metrics(y, yhat)
print("Train accuracy:", round(m["accuracy"], 3))
print("Per class:", m["per_class"])


Example: Sunny / Cool / High / True → **No**. Overcast days strongly favour **Yes**. Train accuracy ≈ 92.9%.

## Additional Q3 — Sentiment (pos / neg)

In [ ]:
SENTIMENT = [
    ("I love this sandwich", "pos"),
    ("This is an amazing place", "pos"),
    ("I feel very good about these beers", "pos"),
    ("This is my best work", "pos"),
    ("What an awesome view", "pos"),
    ("I do not like this restaurant", "neg"),
    ("I am tired of this stuff", "neg"),
    ("I can't deal with this", "neg"),
    ("He is my sworn enemy", "neg"),
    ("My boss is horrible", "neg"),
    ("This is an awesome place", "pos"),
    ("I do not like the taste of this juice", "neg"),
    ("I love to dance", "pos"),
    ("I am sick and tired of this place", "neg"),
    ("What a great holiday", "pos"),
    ("That is a bad locality to stay", "neg"),
    ("We will have good fun tomorrow", "pos"),
    ("I went to my enemy's house today", "neg"),
]

texts = [t for t, _ in SENTIMENT]
labels = [l for _, l in SENTIMENT]

split = 14
clf = TextNB(alpha=1.0).fit(texts[:split], labels[:split])
y_test = labels[split:]
yhat = clf.predict(texts[split:])
m_test = metrics(y_test, yhat)

print("Train size", split, " Test size", len(y_test))
print("Train class counts", dict(clf.class_counts_))
print("Test true", y_test)
print("Test pred", yhat)
print("Test accuracy", round(m_test["accuracy"], 3))
print("Test per-class", m_test["per_class"])

clf_all = TextNB(alpha=1.0).fit(texts, labels)
print("Fit-on-all train accuracy", metrics(labels, clf_all.predict(texts))["accuracy"])

print("\nExtra sentences:")
for s in ["I love this place", "I do not like this", "What a great view", "My enemy is horrible"]:
    pred, post = clf_all.predict_one(s)
    print(f"  {s!r:30s} → {pred:3s}  {fmt_post(post)}")


Hold-out (last 4 sentences) accuracy is only 50% because the test set is tiny. Fit on all 18 examples recovers 100% train accuracy. Extra sentences classify as expected (love/great → pos, not like/enemy/horrible → neg).